In [1]:
# from PIL import Image, ImageDraw
# from IPython.display import display  # để hiển thị trong Jupyter/Kaggle

# def draw_boxes(image_path, boxes, width=2, show=True, save_path=None):
#     """
#     Vẽ khung xanh lá quanh các box và hiển thị trực tiếp.
#     boxes: [((x1,y1),(x2,y2)), ...]
#     """
#     img = Image.open(image_path).convert("RGB")
#     draw = ImageDraw.Draw(img)
#     for (x1, y1), (x2, y2) in boxes:
#         draw.rectangle([x1, y1, x2, y2], outline=(0, 255, 0), width=width)

#     if show:
#         display(img)          # chỉ show, không lưu

#     if save_path:             # tuỳ chọn: muốn lưu thì truyền save_path
#         img.save(save_path, quality=95)

#     return img                # trả về ảnh nếu bạn muốn dùng tiếp

# # Ví dụ dùng:
# image_path = "/kaggle/input/lucifer-kfn-1/_output_/lucifer-kfn1/L21_V001/10353.jpg"
# logo_box = ((437, 14), (505, 47))
# logo2_box = ((3, 2), (55, 34))
# subtitle_box = ((0, 272), (533, 288))
# _ = draw_boxes(image_path, [logo_box, subtitle_box, logo2_box], width=2, show=True)


In [2]:
%%capture
!pip install easyocr opencv-python matplotlib joblib

In [3]:
# import os
# import cv2
# import numpy as np
# import easyocr
# from typing import List, Tuple, Optional
# from tqdm.notebook import tqdm
# from joblib import Parallel, delayed
# import time


# import easyocr
# import cv2
# import numpy as np

# # Tạo đối tượng reader cho nhận diện tiếng Anh
# reader = easyocr.Reader(['en'], gpu=True, detector="craft")

# # Danh sách các đường dẫn ảnh
# paths = [
#     "/kaggle/input/kf-full/L01_V001/0.jpg",
#     "/kaggle/input/kf-full/L01_V001/10157.jpg",
#     "/kaggle/input/kf-full/L01_V001/10199.jpg",
#     "/kaggle/input/kf-full/L01_V003/10094.jpg",
#     "/kaggle/input/kf-full/L01_V003/10241.jpg",
#     "/kaggle/input/kf-full/L01_V003/10262.jpg",
#     "/kaggle/input/kf-full/L01_V003/10458.jpg",
#     "/kaggle/input/kf-full/L01_V003/11942.jpg"
# ]

# # Danh sách chứa các ảnh đã xử lý
# images = []

# # Đọc tất cả các ảnh, xử lý và thêm vào danh sách
# for path in paths:
#     img = cv2.imread(path)
    
#     # Vẽ các hình chữ nhật lên ảnh để bỏ qua vùng không cần nhận diện
#     logo_box = ((520, 40), (595, 75))
#     subtitle_box = ((0, 435), (640, 460))
#     boxes_to_ignore = [logo_box, subtitle_box]
#     for box in boxes_to_ignore:
#         cv2.rectangle(img, box[0], box[1], (0, 0, 0), -1)
    
#     # # Thay đổi kích thước ảnh (nếu cần)
#     width = int(img.shape[1] * 3.0)
#     height = int(img.shape[0] * 3.0)
#     img_resized = cv2.resize(img, (width, height), interpolation=cv2.INTER_CUBIC)
    
#     # Thêm ảnh vào danh sách images
#     images.append(img_resized)
# # Chuyển danh sách ảnh thành numpy array (mảng ba chiều)
# images_np = np.array(images)

# # Gọi detect cho batch ảnh (danh sách numpy array)
# detection_results = reader.detect(images_np, optimal_num_chars=None, reformat=False)

# # In kết quả
# for i, result in enumerate(detection_results):
#     print(f"Detection result for image {i}:")
#     print(result)


In [4]:
import os
import cv2
import numpy as np
import easyocr
from typing import List, Tuple, Optional, Dict
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
import time
import pickle
class ImageClassifier:
    """
    Quét một phần hoặc toàn bộ thư mục ảnh, phân loại chúng thành các nhóm
    CÓ VĂN BẢN và KHÔNG CÓ VĂN BẢN, lưu kết quả ra file và trả về cả hai danh sách.
    """

    def __init__(self, input_path: str, ignore_boxes: List[Tuple], gpu: bool = True, progress: Optional[Dict[str, int]] = None):
        # Phần khởi tạo __init__ không thay đổi
        self.input_path = input_path
        self.ignore_boxes = ignore_boxes
        
        print("Đang quét và sắp xếp tất cả các đường dẫn ảnh...")
        all_image_paths = []
        if not os.path.isdir(self.input_path):
            raise ValueError(f"Đường dẫn đầu vào không tồn tại hoặc không phải là thư mục: {self.input_path}")

        for folder_name in os.listdir(self.input_path):
            folder_path = os.path.join(self.input_path, folder_name)
            if os.path.isdir(folder_path):
                files = [
                    os.path.join(folder_path, file) 
                    for file in os.listdir(folder_path) 
                    if file.endswith('.jpg')
                ]
                all_image_paths.extend(files)
        
        all_image_paths.sort() 
        
        total_images = len(all_image_paths)
        self.start_index = 0
        self.end_index = total_images

        if progress:
            self.start_index = progress.get('start', 0)
            self.end_index = progress.get('end', total_images) 
        print({
            "length": total_images,
            "start": self.start_index,
            "end": self.end_index
        })
        self.start_index = max(0, self.start_index)
        self.end_index = min(total_images, self.end_index)
        if self.start_index >= self.end_index:
             print(f"Cảnh báo: 'start' ({self.start_index}) lớn hơn hoặc bằng 'end' ({self.end_index}). Sẽ không có ảnh nào được xử lý.")
             self.paths_to_process = []
        else:
            self.paths_to_process = all_image_paths[self.start_index:self.end_index]

        print(f"Đã tìm thấy tổng cộng {total_images} ảnh.")
        print(f"Sẽ xử lý {len(self.paths_to_process)} ảnh từ chỉ số {self.start_index} đến {self.end_index}.")

        print("Đang khởi tạo EasyOCR Reader...")
        try:
            self.reader = easyocr.Reader(['en'], gpu=gpu, detector="craft")
            print(f"EasyOCR Reader đã sẵn sàng (GPU: {gpu}).")
        except Exception as e:
            print(f"Không thể khởi tạo EasyOCR với GPU={gpu}, lỗi: {e}. Thử lại với CPU.")
            self.reader = easyocr.Reader(['en'], gpu=False, detector="craft")
            print("EasyOCR Reader đã sẵn sàng (GPU: False).")

    # --- Các hàm trợ giúp không thay đổi ---
    def _read_image_and_path(self, path: str):
        try:
            img = cv2.imread(path)
            return (img, path)
        except Exception as e:
            print(f"💥 Lỗi ngoại lệ khi đọc ảnh {path}: {e}")
            return (None, path)

    def _preprocess_image(self, image, scale_factor=2.0):
        if isinstance(image, str):
            img = cv2.imread(image)
            if img is None:
                print(f"Lỗi: Không thể đọc ảnh từ đường dẫn: {image}")
                return None
        else:
            img = image

        # ignore boxes if image is landscape
        if img.shape[0] < img.shape[1]:
            for box in self.ignore_boxes:
                cv2.rectangle(img, box[0], box[1], (0), -1)
        
        # upscale image
        width = int(img.shape[1] * scale_factor)
        height = int(img.shape[0] * scale_factor)
        upscaled_image = cv2.resize(img, (width, height), interpolation=cv2.INTER_CUBIC)

        # convert BGR to gray
        if len(upscaled_image.shape) == 3 and upscaled_image.shape[2] == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = upscaled_image 

        # enhance gray image
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4,4))
        enhanced_gray = clahe.apply(gray)
        
        processed_image = cv2.cvtColor(enhanced_gray, cv2.COLOR_GRAY2BGR)
        
        return processed_image
    
    def _save_paths_to_file(self, file_path: Optional[str], paths: List[str]):
        """Hàm trợ giúp để lưu một danh sách các đường dẫn ra file."""
        if file_path:
            if paths:
                try:
                    absolute_output_path = os.path.abspath(file_path)
                    print(f"\n💾 Đang lưu {len(paths)} đường dẫn vào file: {absolute_output_path}")
                    with open(file_path, 'w', encoding='utf-8') as f:
                        for path in paths:
                            f.write(f"{path}\n")
                    print("   Lưu file thành công.")
                except Exception as e:
                    print(f"   💥 Lỗi: Không thể lưu file vào {file_path}. Lý do: {e}")
            else:
                print(f"\nℹ️ Không có đường dẫn nào được tìm thấy để lưu vào file '{file_path}'.")

    def _save_paths_to_pkl_file(self, file_path: Optional[str], paths: List[str]):
        """Hàm trợ giúp để lưu một danh sách các đường dẫn ra file pickle (.pkl)."""
        if file_path:
            if paths:
                try:
                    # Đảm bảo file có đuôi .pkl để dễ nhận biết
                    if not file_path.endswith('.pkl'):
                        file_path += '.pkl'
    
                    absolute_output_path = os.path.abspath(file_path)
                    print(f"\n💾 Đang lưu {len(paths)} đường dẫn vào file pickle: {absolute_output_path}")
    
                    # Mở file ở chế độ 'write binary' (wb)
                    with open(file_path, 'wb') as f:
                        # Dùng pickle.dump để lưu toàn bộ đối tượng list vào file
                        pickle.dump(paths, f)
                    print("   Lưu file pickle thành công.")
    
                except Exception as e:
                    print(f"   💥 Lỗi: Không thể lưu file pickle vào {file_path}. Lý do: {e}")
            else:
                print(f"\nℹ️ Không có đường dẫn nào được tìm thấy để lưu vào file pickle '{file_path}'.")

    # --- PHƯƠNG THỨC CHÍNH ĐÃ ĐƯỢC CẬP NHẬT HOÀN CHỈNH ---
    def classify_images(self, 
                        batch_size: int = 64, 
                        with_text_file: Optional[str] = "paths_with_text.pkl", 
                        without_text_file: Optional[str] = "paths_without_text.pkl"
                        ) -> Tuple[List[str], List[str]]:
        """
        Phân loại ảnh thành nhóm có và không có văn bản, lưu kết quả ra file,
        và trả về cả hai danh sách.

        Args:
            batch_size (int): Số lượng ảnh xử lý trong mỗi batch.
            with_text_file (Optional[str]): Tên file để lưu các đường dẫn có văn bản.
            without_text_file (Optional[str]): Tên file để lưu các đường dẫn không có văn bản.

        Returns:
            Tuple[List[str], List[str]]: Một tuple chứa (danh_sách_có_text, danh_sách_không_có_text).
        """
        if not self.paths_to_process:
            print("Không có ảnh nào trong phạm vi được chọn để xử lý.")
            return [], []

        print(f"Bắt đầu phân loại {len(self.paths_to_process)} ảnh...")

        failed_read_paths = []
        paths_with_text = []
        paths_without_text = []

        for i in tqdm(range(0, len(self.paths_to_process), batch_size), desc="Tổng tiến trình"):
            batch_paths = self.paths_to_process[i:i + batch_size]
            
            batch_read_results = [self._read_image_and_path(path) for path in batch_paths]
            valid_images_with_paths = [(img, path) for img, path in batch_read_results if img is not None]
            failed_paths_in_batch = [path for img, path in batch_read_results if img is None]
            failed_read_paths.extend(failed_paths_in_batch)
            if not valid_images_with_paths: 
                continue
            batch_images, current_paths = zip(*valid_images_with_paths)
            processed_images = [self._preprocess_image(img) for img in batch_images]
            if not processed_images: 
                paths_without_text.extend(current_paths)
                continue
            detection_results = self.reader.detect(np.array(processed_images), reformat=False)
            # --- LOGIC PHÂN LOẠI CỐT LÕI ---
            for idx, (h_boxes, f_boxes) in enumerate(zip(*detection_results)):
                if h_boxes or f_boxes:
                    paths_with_text.append(current_paths[idx])
                else:
                    paths_without_text.append(current_paths[idx])
        
        print("\n✅ Hoàn tất việc phân loại ảnh.")
        if failed_read_paths:
            print(f"⚠️ Ghi nhận {len(failed_read_paths)} ảnh bị lỗi khi đọc.")
        
        # --- LƯU CẢ HAI FILE KẾT QUẢ ---
        self._save_paths_to_file(with_text_file.replace('.pkl', '.txt'), paths_with_text)
        self._save_paths_to_file(without_text_file.replace('.pkl', '.txt'), paths_without_text)

        self._save_paths_to_pkl_file(with_text_file, paths_with_text)
        self._save_paths_to_pkl_file(without_text_file, paths_without_text)
        # --- TRẢ VỀ CẢ HAI DANH SÁCH ---
        return paths_with_text, paths_without_text

In [ ]:
DATASET_PATH = "/kaggle/input/lucifer-kfn-1/_output_/lucifer-kfn1"
# logo_box = ((520, 40), (595, 75))
# subtitle_box = ((0, 435), (640, 460))
logo_box = ((437, 14), (505, 47))
logo2_box = ((3, 2), (55, 34))
subtitle_box = ((0, 272), (533, 288))
boxes_to_ignore = [logo_box, logo2_box, subtitle_box]
# progress = {'length': 139457, 'start': 0, 'end': 1000}
progress = {}
classifier = ImageClassifier(input_path=DATASET_PATH, ignore_boxes=boxes_to_ignore, gpu=True, progress=progress)
with_text, without_text = classifier.classify_images(batch_size=64)

print(f"image ratio with text: {len(with_text) / (len(with_text) + len(without_text))}")

Đang quét và sắp xếp tất cả các đường dẫn ảnh...
{'length': 139457, 'start': 0, 'end': 1000}
Đã tìm thấy tổng cộng 139457 ảnh.
Sẽ xử lý 1000 ảnh từ chỉ số 0 đến 1000.
Đang khởi tạo EasyOCR Reader...
EasyOCR Reader đã sẵn sàng (GPU: True).
Bắt đầu phân loại 1000 ảnh...


Tổng tiến trình:   0%|          | 0/16 [00:00<?, ?it/s]


✅ Hoàn tất việc phân loại ảnh.

💾 Đang lưu 305 đường dẫn vào file: /kaggle/working/paths_with_text.txt
   Lưu file thành công.

💾 Đang lưu 695 đường dẫn vào file: /kaggle/working/paths_without_text.txt
   Lưu file thành công.

💾 Đang lưu 305 đường dẫn vào file pickle: /kaggle/working/paths_with_text.pkl
   Lưu file pickle thành công.

💾 Đang lưu 695 đường dẫn vào file pickle: /kaggle/working/paths_without_text.pkl
   Lưu file pickle thành công.
image ratio with text: 0.305
